# ThreatVisionAI

**A Hybrid CNN-ViT Framework for Image-Based Malware Classification**

Demo notebook. Walks through the trained model end to end:

1. Load the three branches (Raw CNN, Wavelet CNN, ViT-Tiny) from saved checkpoints.
2. Run weighted soft voting on the test set with weights 0.50 / 0.40 / 0.10.
3. Show per-class metrics and the ensemble confusion matrix.
4. Visualize the Autorun.K vs Yuner.A failure case with Grad-CAM.

The notebook expects the repository layout described in the README and looks for `data/` and `models/` one level up from this folder. It runs on CPU, although the test pass is faster on GPU.

Companion to the paper accepted to **IEEE World AI IoT Congress (AIIoT) 2026**.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

# Make ../src importable
HERE = Path.cwd()
REPO_ROOT = HERE if (HERE / "src").exists() else HERE.parent
SRC_DIR = REPO_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

DATA_RAW_ROOT  = REPO_ROOT / "data" / "raw"
DATA_WAV_ROOT  = REPO_ROOT / "data" / "wavelet"
MODELS_DIR     = REPO_ROOT / "models"
RESULTS_DIR    = REPO_ROOT / "results"
FIGURES_DIR    = REPO_ROOT / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root  : {REPO_ROOT}")
print(f"Data (raw) : {DATA_RAW_ROOT}  exists={DATA_RAW_ROOT.exists()}")
print(f"Data (wav) : {DATA_WAV_ROOT}  exists={DATA_WAV_ROOT.exists()}")
print(f"Models dir : {MODELS_DIR}     exists={MODELS_DIR.exists()}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

from data_loaders import get_paired_test_loader
from models import build_resnet18, build_vit_tiny, load_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 2. Build the paired test loader

The ensemble needs raw and wavelet inputs for the same underlying sample. `PairedRawWaveletDataset` aligns the two folders by relative path and returns `(raw_img, wav_img, label)` triples.

In [ ]:
DATA_AVAILABLE = DATA_RAW_ROOT.exists() and DATA_WAV_ROOT.exists()

if DATA_AVAILABLE:
    test_loader, classes = get_paired_test_loader(
        str(DATA_RAW_ROOT), str(DATA_WAV_ROOT),
        batch_size=128, num_workers=2,
    )
    num_classes = len(classes)
    print(f"Paired test samples: {len(test_loader.dataset)}")
    print(f"Classes ({num_classes}): {classes}")
else:
    # Fallback: hardcoded 25-class Malimg order so the rest of the notebook still loads.
    from gradcam import MALIMG_CLASSES_25
    classes = MALIMG_CLASSES_25
    num_classes = len(classes)
    test_loader = None
    print("Dataset not found locally. Skipping test pass; only the Grad-CAM section needs raw data.")

## 3. Load the three trained branches

If a checkpoint is missing, that branch is skipped and the ensemble falls back to the available branches with renormalized weights.

In [ ]:
def try_load_branch(builder, ckpt_name, name):
    path = MODELS_DIR / ckpt_name
    if not path.exists():
        print(f"[skip] {name}: no checkpoint at {path}")
        return None
    model = builder(num_classes)
    load_checkpoint(model, str(path), device)
    model = model.to(device).eval()
    print(f"[load] {name}: {path.name}")
    return model

raw_model = try_load_branch(build_resnet18, "best_resnet18.pth",        "Raw CNN")
wav_model = try_load_branch(build_resnet18, "best_wavelet_resnet18.pth", "Wavelet CNN")
try:
    vit_model = try_load_branch(build_vit_tiny, "best_vit_tiny_raw.pth", "ViT-Tiny")
except Exception as e:
    print(f"[skip] ViT-Tiny: {e}")
    vit_model = None

## 4. Weighted soft voting on the test set

Per-branch softmax probabilities are combined with the paper weights:

$$P_{\text{ensemble}} = 0.50 \cdot P_{\text{raw}} + 0.40 \cdot P_{\text{wav}} + 0.10 \cdot P_{\text{vit}}$$

If the ViT is unavailable, the two CNN weights renormalize to 0.50/0.90 and 0.40/0.90.

In [ ]:
W_RAW, W_WAV, W_VIT = 0.50, 0.40, 0.10

@torch.no_grad()
def collect_probs():
    raw_ps, wav_ps, vit_ps, ys = [], [], [], []
    for x_raw, x_wav, y in test_loader:
        x_raw = x_raw.to(device)
        x_wav = x_wav.to(device)
        if raw_model is not None:
            raw_ps.append(torch.softmax(raw_model(x_raw), dim=1).cpu())
        if wav_model is not None:
            wav_ps.append(torch.softmax(wav_model(x_wav), dim=1).cpu())
        if vit_model is not None:
            vit_ps.append(torch.softmax(vit_model(x_raw), dim=1).cpu())
        ys.extend(y.numpy())
    out = {}
    if raw_ps: out["raw"] = torch.cat(raw_ps)
    if wav_ps: out["wav"] = torch.cat(wav_ps)
    if vit_ps: out["vit"] = torch.cat(vit_ps)
    return out, np.array(ys)

if test_loader is not None and raw_model is not None and wav_model is not None:
    probs, y_true = collect_probs()

    has_vit = "vit" in probs
    if has_vit:
        fused = (W_RAW * probs["raw"]
                 + W_WAV * probs["wav"]
                 + W_VIT * probs["vit"])
        weight_str = f"raw=0.50, wav=0.40, vit=0.10"
    else:
        denom = W_RAW + W_WAV
        fused = (W_RAW / denom) * probs["raw"] + (W_WAV / denom) * probs["wav"]
        weight_str = f"raw=0.556, wav=0.444 (ViT unavailable)"

    from sklearn.metrics import accuracy_score, f1_score
    def metrics(p):
        preds = p.argmax(1).numpy()
        return (
            accuracy_score(y_true, preds),
            f1_score(y_true, preds, average="weighted"),
            f1_score(y_true, preds, average="macro"),
        )

    print(f"{'Branch':<14} {'Acc':>8} {'WeightedF1':>12} {'MacroF1':>10}")
    print("-" * 46)
    for name, key in [("Raw CNN", "raw"), ("Wavelet CNN", "wav"), ("ViT-Tiny", "vit")]:
        if key in probs:
            a, fw, fm = metrics(probs[key])
            print(f"{name:<14} {a:>8.4f} {fw:>12.4f} {fm:>10.4f}")
    a, fw, fm = metrics(fused)
    print("-" * 46)
    print(f"{'Ensemble':<14} {a:>8.4f} {fw:>12.4f} {fm:>10.4f}")
    print(f"Weights: {weight_str}")
else:
    print("Test pass skipped (need data and at least Raw+Wavelet checkpoints).")

## 5. Per-class report

The Autorun.K family is the dominant failure case: its samples are systematically misclassified as Yuner.A because the two are visually near-identical when rendered as bytecode images. F1 for Autorun.K is reported as 0.0 in the paper.

In [ ]:
if test_loader is not None and raw_model is not None and wav_model is not None:
    from sklearn.metrics import classification_report
    fused_preds = fused.argmax(1).numpy()
    print(classification_report(y_true, fused_preds, target_names=classes, digits=4, zero_division=0))

## 6. Ensemble confusion matrix

In [ ]:
if test_loader is not None and raw_model is not None and wav_model is not None:
    from utils import plot_confusion_matrix
    cm_path = RESULTS_DIR / "demo_ensemble_confusion_matrix.png"
    plot_confusion_matrix(
        y_true, fused_preds, classes, str(cm_path),
        title="Ensemble Test Confusion Matrix",
    )
    from IPython.display import Image, display
    display(Image(filename=str(cm_path)))
    print(f"Saved to: {cm_path}")

## 7. Grad-CAM: Autorun.K vs Yuner.A

Grad-CAM applied to the last convolutional layer of the Raw CNN. The same regions of attention activate for both families, which is consistent with the observation that the bytecode-image representation does not separate them well.

In [ ]:
if raw_model is None:
    print("Raw CNN checkpoint not available; skipping Grad-CAM.")
elif not DATA_AVAILABLE:
    print("Raw test data not available; skipping Grad-CAM.")
else:
    from gradcam import make_autorun_vs_yuner_figure

    def first_png_in(family):
        family_dir = DATA_RAW_ROOT / "test" / family
        for f in sorted(os.listdir(str(family_dir))):
            if f.endswith(".png"):
                return str(family_dir / f)
        raise FileNotFoundError(family_dir)

    autorun_path = first_png_in("Autorun.K")
    yuner_path   = first_png_in("Yuner.A")
    fig_path     = FIGURES_DIR / "demo_gradcam_autorun_yuner.png"

    make_autorun_vs_yuner_figure(
        raw_model, device, autorun_path, yuner_path, str(fig_path),
    )

    from IPython.display import Image, display
    display(Image(filename=str(fig_path)))

## Summary

The demo reproduces the paper's headline numbers from saved checkpoints, shows the Autorun.K limitation in the per-class report, and produces the Grad-CAM figure used in the interpretability discussion. To run any of the underlying steps as standalone scripts (training a branch, FGSM sweep, full evaluation), use the `src/` modules directly. See the README for command-line examples.